# Naive Bayes Baseline

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
import time
import joblib

# 🔎 Model Comparison Phase 
(Compare all models)- check the performance before choosing the best one.

In [ ]:
def safe_evaluate(csv_path, dataset_name, model, model_name):

    df = pd.read_csv(csv_path)
    X = df["text"].astype(str)
    y = df["label"].astype(str)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("clf", model)
    ])

    pipeline.fit(X_train, y_train)

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    train_f1 = f1_score(y_train, train_pred, average="weighted")
    test_f1 = f1_score(y_test, test_pred, average="weighted")

    return {
        "Dataset": dataset_name,
        "Model": model_name,
        "Train Accuracy": round(train_acc, 4),
        "Test Accuracy": round(test_acc, 4),
        "Train F1": round(train_f1, 4),
        "Test F1": round(test_f1, 4)
    }


# MODELS
models = [
    ("Naive Bayes", MultinomialNB()),
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("SVM (Linear)", LinearSVC()),
    ("Random Forest", RandomForestClassifier())
]


# RUN ALL MODELS
results = []
for name, model in models:

    # BEFORE AUGMENTATION
    results.append(
        safe_evaluate(
            "intent_classification_datasets/data_new.csv",
            "Before Augmentation",
            model,
            name
        )
    )

    # AFTER AUGMENTATION
    results.append(
        safe_evaluate(
            "intent_classification_datasets/data_new_augmented_dataset.csv",
            "After Augmentation",
            model,
            name
        )
    )

# FINAL TABLE
df_results = pd.DataFrame(results)

print("\n" + "=" * 100)
print("📊 ALL MODELS RESULTS")
print("=" * 100)
print(df_results)


# BEST MODEL
best_model = df_results.loc[
    df_results["Test Accuracy"].idxmax()
]

print("\n" + "=" * 100)
print("🏆 BEST MODEL")
print("=" * 100)

print(best_model)

# ↩ BEFORE AUGMENTATION — Naive Bayes Baseline

In [ ]:
# Load dataset
df_before = pd.read_csv(
    "intent_classification_datasets/data_new.csv",
    encoding='utf-8-sig',
    on_bad_lines='skip'
)

X = df_before["text"]
y = df_before["label"]

#Load dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model Pipeline
model_before = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("nb", MultinomialNB())
])

# Train
model_before.fit(X_train, y_train)

# Predict
y_train_pred = model_before.predict(X_train)
y_test_pred = model_before.predict(X_test)

# Evaluation
print(" \n ========================= BEFORE AUGMENTATION =========================  ")
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy :", accuracy_score(y_test, y_test_pred))
print("\n ========================= ")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

# ↪ AFTER AUGMENTATION — Enhanced Dataset Model 

In [2]:
df_after = pd.read_csv(
    "intent_classification_datasets/data_new_augmented_dataset.csv",
    encoding='utf-8-sig',
    on_bad_lines='skip'
)

X = df_after["text"]
y = df_after["label"]


# 70% Train - 30% مؤقت
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# تقسيم الـ 30% إلى Validation و Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

model_after = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("nb", MultinomialNB())
])

model_after.fit(X_train, y_train)
y_train_pred = model_after.predict(X_train)
y_test_pred = model_after.predict(X_test)
y_val_pred = model_after.predict(X_val)


print("\n ========================= AFTER AUGMENTATION =========================")
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy :", accuracy_score(y_test, y_test_pred))
print("\n ========================= ")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))


 ========================= AFTER AUGMENTATION =========================
Train Accuracy: 0.9965796579657966
Test Accuracy : 0.9958018471872376


Classification Report:
                           precision    recall  f1-score   support

                complaint       1.00      0.99      1.00       150
         general_question       0.99      1.00      1.00       149
                logistics       1.00      1.00      1.00       103
                  payment       0.98      0.99      0.99       115
          product_inquiry       1.00      0.99      1.00       131
                   refund       0.99      1.00      0.99       138
self_harm_or_suicide_risk       1.00      1.00      1.00       150
        technical_support       1.00      0.99      1.00       150
                  unknown       1.00      0.99      1.00       105

                 accuracy                           1.00      1191
                macro avg       1.00      1.00      1.00      1191
             weighted avg 

In [ ]:
print("validation accuracy :", accuracy_score(y_val,y_val_pred))

In [ ]:
start_time = time.time()
test_pred = model_after.predict(X_test)
end_time = time.time()

total_latency = end_time - start_time
avg_latency = total_latency / len(X_test)

print(f"Total Inference Time: {total_latency:.6f} sec")
print(f"Average Latency per Sample: {avg_latency:.6f} sec/sample")

In [ ]:
latencies = []
# قياس زمن التنبؤ لكل عينة
for text in X_test:
    
    start_time = time.perf_counter()

    model_after.predict([text])

    end_time = time.perf_counter()

    latency_ms = (end_time - start_time) * 1000
    latencies.append(latency_ms)

# حساب الإحصائيات
avg_latency = np.mean(latencies)
p95_latency = np.percentile(latencies, 95)
p99_latency = np.percentile(latencies, 99)
min_latency = np.min(latencies)
max_latency = np.max(latencies)

# التقرير
print("\n==========================================================")
print("             INFERENCE LATENCY REPORT           ")
print("==========================================================")
print(f"Model                 : Multinomial Naive Bayes")
print(f"Total Queries Tested  : {len(X_test)}")
print(f"Minimum Latency       : {min_latency:.2f} ms")
print(f"Average Latency (Mean): {avg_latency:.2f} ms")
print(f"95th Percentile (P95) : {p95_latency:.2f} ms")
print(f"99th Percentile (P99) : {p99_latency:.2f} ms")
print(f"Maximum Latency       : {max_latency:.2f} ms")
print(f"Throughput Capability : {int(1000 / avg_latency)} requests/second (approx)")
print("==========================================================\n")

In [ ]:
joblib.dump(model_after, "nb_after_model.pkl")
print("Model saved: nb_after_model.pkl")

# 🤖 MODEL TESTING (LOAD + INTERACTIVE DEMO)

In [ ]:
# Load trained model
model_after = joblib.load("nb_after_model.pkl")

In [ ]:
# Interactive testing
while True:
    text = input("\nEnter text to classify (type 'exit' to quit): ")
    if text.lower() == "Exit":
        break
    pred = model_after.predict([text])[0]
    proba = model_after.predict_proba([text]).max()

    print(f"Predicted Intent: {pred}")
    print(f"Confidence: {proba:.2%}")